# LP 和 LCS：共享提升坐标与形式伴随
对应 `def:lp`、`app:lp-examples`、`app:lcs-details`。

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "qldpc_generator").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
import numpy as np
from qldpc_generator.algebra import x, y


形式伴随同时转置矩阵并反转循环方向：$$M^*=\overline M^{\mathsf T},\qquad\mathbb B(M^*)=\mathbb B(M)^{\mathsf T}.\tag{1}$$

In [2]:
import sympy as sp
from qldpc_generator.algebra import formal_adjoint,cyclic_coefficients,cyclic_group,expand_group_matrix
M=sp.Matrix([[1+x,x**2,0],[1,0,1+x**3]])
g=cyclic_group(5)
assert np.array_equal(expand_group_matrix(g,cyclic_coefficients(formal_adjoint(M),5)),expand_group_matrix(g,cyclic_coefficients(M,5)).T)
from qldpc_generator.paper_cases import cyclic_lp_case
for lift, expected in [(16,80),(21,100),(30,136)]:
    code=cyclic_lp_case(lift)
    assert code.n==34*lift and code.k==expected
    print(lift,code.parameters())

16 {'n': 544, 'k': 80, 'rank_x': 232, 'rank_z': 232, 'checks_x': 240, 'checks_z': 240, 'max_row_weight': 8, 'max_column_weight': 5}
21 {'n': 714, 'k': 100, 'rank_x': 307, 'rank_z': 307, 'checks_x': 315, 'checks_z': 315, 'max_row_weight': 8, 'max_column_weight': 5}
30 {'n': 1020, 'k': 136, 'rank_x': 442, 'rank_z': 442, 'checks_x': 450, 'checks_z': 450, 'max_row_weight': 8, 'max_column_weight': 5}


以上独立复算 N、K 和两个二元秩。论文所列 12、16、20 为文献报告的距离上界，此处不把它们当成精确距离或已生成的证书。

In [3]:
from qldpc_generator import lcs,lifted_product,hgp
from qldpc_generator.algebra import polynomial_to_circulant
code=lcs(2,4)
assert code.hx.shape==code.hz.shape==(24,52)
assert code.k==4
result=code.distance(4)
assert result['status']=='exact' and result['value']==4
result

{'status': 'exact',
 'value': 4,
 'sectors': {'X': {'status': 'exact',
   'value': 4,
   'lower_bound': 4,
   'witness': [32, 33, 34, 35],
   'subsets': 1379},
  'Z': {'status': 'exact',
   'value': 4,
   'lower_bound': 4,
   'witness': [32, 33, 34, 35],
   'subsets': 1379}}}

In [4]:
shared=lifted_product([[1+x]],[[1+x**2]],3)
ordinary=hgp(polynomial_to_circulant(1+x,3),polynomial_to_circulant(1+x**2,3).T)
assert (shared.n,ordinary.n)==(6,18)
print('shared lift N=',shared.n,'; two independent lift coordinates N=',ordinary.n)

shared lift N= 6 ; two independent lift coordinates N= 18


LCS 的一般距离式 $D=\min(L,2\ell+1)$ 仍是猜想。这里仅证明给定 `[[52,4,4]]` 有限实例。